# Interest Rate Risk Management in Colab

This notebook is a Colab-friendly variant of the local Interest Rate Risk demo.

Learning goals:

- generate a synthetic banking portfolio
- compare exposure by account type
- estimate how a rate shock changes annual interest cost
- explain which product category drives repricing pressure


In [ ]:
%pip install faker pandas matplotlib seaborn -q

In [ ]:
import random
import datetime as dt
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from faker import Faker

fake = Faker('en_IN')
Faker.seed(42)
random.seed(42)
sns.set_theme(style='whitegrid')

ACCOUNT_TYPES = {
    'Savings': {'rate': (3.0, 4.0), 'balance': (1_000, 500_000)},
    'Current': {'rate': (0.1, 0.5), 'balance': (10_000, 10_000_000)},
    'Fixed Deposit': {'rate': (5.5, 7.0), 'balance': (50_000, 50_000_000)},
    'Recurring Deposit': {'rate': (5.0, 6.5), 'balance': (500, 1_000_000)}
}

def random_date(years_back=10):
    end = dt.date.today()
    start = end - dt.timedelta(days=365 * years_back)
    return fake.date_between(start_date=start, end_date=end)

def build_portfolio(num_rows=5000):
    rows = []
    for idx in range(1, num_rows + 1):
        account_type = random.choice(list(ACCOUNT_TYPES.keys()))
        cfg = ACCOUNT_TYPES[account_type]
        rows.append({
            'CustomerID': idx,
            'AccountType': account_type,
            'Balance': round(random.uniform(*cfg['balance']), 2),
            'InterestRate': round(random.uniform(*cfg['rate']), 2),
            'AccountOpenDate': random_date(15),
            'LastTransactionDate': random_date(1)
        })
    return pd.DataFrame(rows)

df = build_portfolio(6000)
df.head()

In [ ]:
shock_bps = 100
shock_pct = shock_bps / 100

df['OriginalInterest'] = df['Balance'] * df['InterestRate'] / 100
df['NewInterest'] = df['Balance'] * (df['InterestRate'] + shock_pct) / 100
df['InterestChange'] = df['NewInterest'] - df['OriginalInterest']

summary = df.groupby('AccountType').agg(
    Accounts=('AccountType', 'size'),
    Exposure=('Balance', 'sum'),
    AvgRate=('InterestRate', 'mean'),
    ShockDelta=('InterestChange', 'sum')
).sort_values('Exposure', ascending=False)

summary.style.format({
    'Exposure': '₹{:,.0f}',
    'AvgRate': '{:.2f}%',
    'ShockDelta': '₹{:,.0f}'
})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df['InterestRate'], bins=30, kde=True, ax=axes[0], color='#0ea5e9')
axes[0].set_title('Interest Rate Distribution')
axes[0].set_xlabel('Interest Rate (%)')

summary['Exposure'].plot(kind='bar', ax=axes[1], color='#2563eb')
axes[1].set_title('Exposure by Account Type')
axes[1].set_ylabel('Exposure (INR)')

plt.tight_layout()
plt.show()

In [ ]:
total_original = df['OriginalInterest'].sum()
total_new = df['NewInterest'].sum()
delta = df['InterestChange'].sum()
largest_driver = summary['ShockDelta'].idxmax()

print(f'Total Interest at current rates: ₹{total_original:,.2f}')
print(f'Total Interest after {shock_bps} bps shock: ₹{total_new:,.2f}')
print(f'Change in annual interest: ₹{delta:,.2f}')
print(f'Largest repricing driver: {largest_driver}')